In [2]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df_bt = pd.read_csv("files/fineTuned_filtered_high_similarity.csv",
                    usecols=["translated_welsh", "translated_welsh_predicted_cefr"])\
         .rename(columns={
             "translated_welsh": "text",
             "translated_welsh_predicted_cefr": "cefr_level"
         })

In [4]:
# Add missing columns with specific default values
df_bt["title"] = "Back-translated A1/A2 sample"
df_bt["lang"] = "cy"
df_bt["source_name"] = "back_translation_pipeline"
df_bt["format"] = "text"
df_bt["category"] = "general"
df_bt["license"] = "CC-BY-SA" 

In [5]:
df_bt

,text,cefr_level,title,lang,source_name,format,category,license
0,"Ac mae'r tywysog yn mynd i ffwrdd, yn ddryslyd.",A1,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
1,"Dyma, i mi, y mae'r cariad yn drist a'r lleafa...",A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
2,Roedd y pumed yn rhyfedd iawn.,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
3,"I fod yn onest, erbyn hyn doeddwn i ddim wir w...",A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
4,Mae'r ffilm yn cyfeirio at ei pwyntiau da.,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
...,...,...,...,...,...,...,...,...
797,3 ... B5 yn aml yn chwarae.,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
798,Mae myfyrwyr o'r Eidal a'r Almaen yn helpu Sae...,A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
799,"Ar ôl un flwyddyn, agorodd y llyfrau hedfan eto.",A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA
800,"Roedd hi'n unig blentyn, geni yn Llundain.",A2,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,CC-BY-SA


In [7]:
# Count how many samples
df_bt["cefr_level"].value_counts()

cefr_level
A2    474
A1    323
B1      5
Name: count, dtype: int64

In [9]:
# Load Welsh CEFR dataset from HuggingFace
ds_welsh = load_dataset("UniversalCEFR/learn_welsh_cy")["train"].to_pandas()  

# Load your B2 JSON data
df_b2 = pd.read_json("files/b2_welsh.json")
df_b2["cefr_level"] = "B2"

In [10]:
ds_welsh

,title,lang,source_name,format,category,cefr_level,license,text
0,Uned 1 - Sgwrs 1,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: Helô, Eryl dw i. Pwy dych chi?\nB: Bore da,..."
1,Uned 1 - Sgwrs 2,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: O na, yr heddlu! (Stopio'r car)\nB: Hello, ..."
2,Uned 1 - Sgwrs 3,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"A: Bore da. Sut dych chi?\nB: Iawn, ond wedi b..."
3,Uned 2 - Sgwrs 1,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,"Ceri: Noswaith dda, Eryl. Sut wyt ti?\nEryl: D..."
4,Uned 2 - Sgwrs 2,cy,mynediad-de-learnwelsh,dialogue-level,reference,A1,public,A: Bore da.\nB: Hmff.\nA: Sut dych chi heddiw?...
...,...,...,...,...,...,...,...,...
1367,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allech chi gyrraedd yn gynnar?
1368,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allet ti gyrraedd yn gynnar?
1369,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allai hi gyrraedd yn gynnar?
1370,Uned 23 - na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Allen nhw gyrraedd yn gynnar?


In [11]:
df_merged = pd.concat([ds_welsh, df_b2, df_bt], ignore_index=True)
df_merged = df_merged.drop_duplicates(subset="text", keep="first").sample(frac=1, random_state=42)

In [12]:
df_merged

,title,lang,source_name,format,category,cefr_level,license,text
1094,Uned 8 - Mynd at,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Pryd rwyt ti'n mynd at y deintydd?
2517,Back-translated A1/A2 sample,cy,back_translation_pipeline,text,general,A2,CC-BY-SA,Rydw i wedi byw yn Japan yn ogystal â Canada.
1479,Uned 4 - Darllen 1 - Carolyn Hitt yn Ateb y Ga...,cy,uwch-1-learnwelsh,sentence-level,reference,B2,public,Dw i'n hoffi gwisgo dillad cysurus.
1442,Uned 3 - dril 2,cy,uwch-1-learnwelsh,sentence-level,reference,B2,public,Ble aiff hi?
1108,Uned 9 - am,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,'Dyn ni wedi bod yn chwilio amdanoch chi.
...,...,...,...,...,...,...,...,...
1644,Uned 11 - Holiadur 4,cy,uwch-1-learnwelsh,sentence-level,reference,B2,public,Pwy sy'n rhedeg gyflyma yn eich teulu / gweith...
1098,Uned 9 - am,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Beth am Bethan?
1133,Uned 10 - Wnei di?,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Wnei di helpu gyda'r gwaith?
1297,Uned 17 -na,cy,sylfaen-de-learnwelsh,sentence-level,reference,A2,public,Caerdydd yw'r ddinas fwya swnllyd.


In [13]:
df_merged["cefr_level"].value_counts()

cefr_level
A1    1085
A2    1080
B2     652
B1       5
Name: count, dtype: int64

In [15]:
df_merged = df_merged[df_merged["cefr_level"] != "B1"]

In [16]:
df_merged["cefr_level"].value_counts()

cefr_level
A1    1085
A2    1080
B2     652
Name: count, dtype: int64

In [17]:
hf_dataset=Dataset.from_pandas(df_merged.reset_index(drop=True))

In [18]:
hf_dataset

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text'],
    num_rows: 2817
})

In [19]:
CEFR_LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}
labels = np.array([label2id[l] for l in hf_dataset["cefr_level"]])

In [20]:
model_name = "EuroBERT/EuroBERT-210m"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

In [21]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [22]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [23]:
# Cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

In [24]:
best_f1 = 0.0
best_trainer = None
best_tokenizer = None

for fold, (train_idx, val_idx) in enumerate(skf.split(hf_dataset, labels), start=1):
    print(f"\n Running Fold {fold}...")

    ds_train = hf_dataset.select(train_idx)
    ds_val = hf_dataset.select(val_idx)

    tok_train = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)
    tok_val = ds_val.map(preprocess, batched=True, remove_columns=ds_val.column_names)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(CEFR_LEVELS),trust_remote_code=True)

    args = TrainingArguments(
        output_dir=f"./eurobert_cefr_welsh_DA/fold_{fold}",  
        num_train_epochs=3, 
        per_device_train_batch_size=2,              
        per_device_eval_batch_size=3,                
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_weighted_f1",
        greater_is_better=True,
        seed=42,
        learning_rate=3.6e-5,
        warmup_ratio=0.1,
        gradient_accumulation_steps=16,      
        optim="adamw_torch_fused",                   
        lr_scheduler_type="linear",                  
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    # Track best trainer
    if metrics["eval_weighted_f1"] > best_f1:
        best_f1 = metrics["eval_weighted_f1"]
        best_trainer = trainer
        best_tokenizer = tokenizer

    # Store fold metrics
    row = {
        "Fold": fold,
        "All CEFR Levels Precision": metrics.get("eval_weighted_precision", 0.0),
        "All CEFR Levels Recall": metrics.get("eval_weighted_recall", 0.0),
        "All CEFR Levels F1": metrics.get("eval_weighted_f1", 0.0),
    }
    for level in CEFR_LEVELS:
        row[f"{level} Precision"] = metrics.get(f"eval_{level}_precision", 0.0)
        row[f"{level} Recall"] = metrics.get(f"eval_{level}_recall", 0.0)
        row[f"{level} F1"] = metrics.get(f"eval_{level}_f1", 0.0)

    all_results.append(row)


 Running Fold 1...


Map: 100%|██████████| 564/564 [00:00<00:00, 11386.76 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_34112\582012621.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,1.100400,1.186650,0.388298,0.221625,0.404145,0.388298,0.386809,1.000000,0.557841,0.666667,0.009259,0.018265,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.941500,0.915844,0.597518,0.527739,0.777326,0.597518,0.925926,0.576037,0.710227,0.492991,0.976852,0.655280,0.000000,0.000000,0.000000,1.000000,0.007634,0.015152,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.648000,0.688600,0.695035,0.687456,0.702173,0.695035,0.863636,0.788018,0.824096,0.607774,0.796296,0.689379,0.000000,0.000000,0.000000,0.590361,0.374046,0.457944,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



 Running Fold 2...


...

Map: 100%|██████████| 563/563 [00:00<00:00, 16787.71 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_34112\582012621.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,1.160300,0.921052,0.547069,0.508865,0.556913,0.547069,0.542005,0.921659,0.682594,0.609524,0.296296,0.398754,0.000000,0.000000,0.000000,0.494382,0.338462,0.401826,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.746800,0.708430,0.698046,0.691317,0.714949,0.698046,0.857955,0.695853,0.768448,0.625418,0.865741,0.726214,0.000000,0.000000,0.000000,0.625000,0.423077,0.504587,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.344700,0.619111,0.742451,0.741489,0.745569,0.742451,0.809524,0.783410,0.796253,0.689796,0.782407,0.733189,0.000000,0.000000,0.000000,0.731481,0.607692,0.663866,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



 Running Fold 5...


Map: 100%|██████████| 563/563 [00:00<00:00, 18745.23 examples/s]
Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\c24082331\AppData\Local\Temp\ipykernel_34112\582012621.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,1.140500,1.037658,0.481350,0.456384,0.484357,0.481350,0.524390,0.594470,0.557235,0.442857,0.574074,0.500000,0.000000,0.000000,0.000000,0.486486,0.138462,0.215569,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.919500,0.947663,0.587922,0.515201,0.680800,0.587922,0.623288,0.838710,0.715128,0.546468,0.680556,0.606186,0.000000,0.000000,0.000000,1.000000,0.015385,0.030303,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.598800,0.770773,0.662522,0.647906,0.664693,0.662522,0.773869,0.709677,0.740385,0.607509,0.824074,0.699411,0.000000,0.000000,0.000000,0.577465,0.315385,0.407960,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [25]:
# Save best-performing model from all folds
final_path = "./eurobert_cefr_welsh_DA_predFiltered/best_model"
best_trainer.save_model(final_path)
best_tokenizer.save_pretrained(final_path)
best_trainer.state.save_to_json(os.path.join(final_path, "trainer_state.json"))

In [26]:
# Convert to DataFrame
df = pd.DataFrame(all_results)

# Compute average row
average_row = df.drop(columns=["Fold"]).mean(numeric_only=True)
average_row["Fold"] = "Average"
df = pd.concat([df, pd.DataFrame([average_row])], ignore_index=True)

# Restructure columns
columns = [("Fold", "")] + [
    ("All CEFR Levels", "Precision"), ("All CEFR Levels", "Recall"), ("All CEFR Levels", "F1"),
    ("A1", "Precision"), ("A1", "Recall"), ("A1", "F1"),
    ("A2", "Precision"), ("A2", "Recall"), ("A2", "F1"),
    ("B1", "Precision"), ("B1", "Recall"), ("B1", "F1"),
    ("B2", "Precision"), ("B2", "Recall"), ("B2", "F1"),
    ("C1", "Precision"), ("C1", "Recall"), ("C1", "F1"),
    ("C2", "Precision"), ("C2", "Recall"), ("C2", "F1"),
]


df = df[[col[0] if col[1] == "" else f"{col[0]} {col[1]}" for col in columns]]
df.columns = pd.MultiIndex.from_tuples(columns)


In [27]:
df

Fold All CEFR Levels                            A1                      \
                 Precision    Recall        F1 Precision    Recall        F1   
0        1        0.702173  0.695035  0.687456  0.863636  0.788018  0.824096   
1        2        0.766581  0.760638  0.762892  0.858537  0.811060  0.834123   
2        3        0.697296  0.701599  0.698529  0.788136  0.857143  0.821192   
3        4        0.745569  0.742451  0.741489  0.809524  0.783410  0.796253   
4        5        0.664693  0.662522  0.647906  0.773869  0.709677  0.740385   
5  Average        0.715262  0.712449  0.707654  0.818740  0.789862  0.803210   

         A2                      ...   B1        B2                      \
  Precision    Recall        F1  ...   F1 Precision    Recall        F1   
0  0.607774  0.796296  0.689379  ...  0.0  0.590361  0.374046  0.457944   
1  0.765258  0.754630  0.759907  ...  0.0  0.616438  0.687023  0.649819   
2  0.686567  0.638889  0.661871  ...  0.0  0.563492  0.546154  0.554688   
3  0.689796  0.782407  0.733189  ...  0.0  0.731481  0.607692  0.663866   
4  0.607509  0.824074  0.699411  ...  0.0  0.577465  0.315385  0.407960   
5  0.671381  0.759259  0.708751  ...  0.0  0.615848  0.506060  0.546855   

         C1                    C2              
  Precision Recall   F1 Precision Recall   F1  
0       0.0    0.0  0.0       0.0    0.0  0.0  
1       0.0    0.0  0.0       0.0    0.0  0.0  
2       0.0    0.0  0.0       0.0    0.0  0.0  
3       0.0    0.0  0.0       0.0    0.0  0.0  
4       0.0    0.0  0.0       0.0    0.0  0.0  
5       0.0    0.0  0.0       0.0    0.0  0.0  

[6 rows x 22 columns]